# Experimentos de Pi-zero (pi0) sobre LIBERO

Corre pi0 (LeRobot) en varias tareas de LIBERO y muestra los resultados (tabla,
tasa de éxito y videos). Es el mismo framework que el notebook de OpenVLA
(`core`, `benchmarks`, `models`); aquí solo cambia el **modelo**.

> Requiere el entorno `pizero` (ver `requirements/environment-pizero.yml`), con **GPU** y el
> checkpoint `lerobot/pi0_libero_finetuned` descargado en `HF_HOME`.


## 1. Setup


In [1]:
import os, sys, glob

# Opcion A (frontera de proceso): este notebook es el CLIENTE del benchmark y
# corre en el entorno `openvla` (robosuite 1.4.1, numpy<2). pi0 (lerobot/torch)
# vive en OTRO proceso (models/Pi_zero/server.py, entorno `pizero`) y se consume
# por HTTP. Por eso aqui NO se instala ni se importa torch/lerobot, y no hace
# falta el shim de torch.load: eso lo hace el servidor al cargar el checkpoint.

os.environ.setdefault("MUJOCO_GL", "egl")     # render headless (sin pantalla)


def _find_project_root():
    """Sube desde el cwd buscando la raiz del proyecto (simulation.py + core/)."""
    d = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, "simulation.py")) and            os.path.isdir(os.path.join(d, "core")):
            return d
        d = os.path.dirname(d)
    hits = glob.glob(os.path.join(os.getcwd(), "**", "simulation.py"), recursive=True)
    if hits:
        return os.path.dirname(os.path.abspath(hits[0]))
    raise RuntimeError("No encontre la raiz del proyecto (simulation.py + core/).")


ROOT = _find_project_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", os.path.join(ROOT, "hf_cache"))   # cache de pesos
print("Proyecto:", ROOT)
print("HF_HOME :", os.environ["HF_HOME"])


Proyecto: /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation
HF_HOME : /home/diegoftpxd/MuJoCo-simulation/hf_cache


## 2. Configuración


In [2]:
from dataclasses import dataclass


@dataclass
class Config:
    suite: str = "libero_10"
    num_tasks: int = 3             # cuantos escenarios (tareas) probar
    episodes_per_task: int = 1     # configuraciones iniciales por tarea
    max_steps: int = 520           # libero_10 es de horizonte largo (oficial ~520)
    out_dir: str = "output/experiments_pizero"


cfg = Config()
cfg


Config(suite='libero_10', num_tasks=3, episodes_per_task=1, max_steps=520, out_dir='output/experiments_pizero')

## 3. Cargar el modelo (una sola vez)


In [3]:
from models.serving import RemoteModel

# pi0 corre como servidor aparte (lo levanta scriptExperiment.sh con MODEL=pi0).
# El notebook es solo cliente: misma interfaz Model, sin deps del modelo.
model = RemoteModel(url="http://localhost:9000")   # 9000 = servidor de pi0
print("cliente conectado ->", model.url)


cliente conectado -> http://localhost:9000


## 4. Correr los experimentos
Un `LiberoController` por tarea (escenario); `run_experiments` corre los
episodios y graba un video por cada uno. pi0 devuelve un chunk de acciones que
el runner ejecuta una a una.


In [4]:
from core import run_experiments
from benchmarks.libero import LiberoController

task_list = LiberoController.tasks(cfg.suite)[:cfg.num_tasks]
benchmarks = [LiberoController(task_id=tid, suite=cfg.suite) for tid, _ in task_list]

results = run_experiments(
    model, benchmarks,
    max_steps=cfg.max_steps,
    episodes_per_task=cfg.episodes_per_task,
    out_dir=cfg.out_dir,
    record_view=View.AGENT,
)
print(f"{len(results)} episodios corridos")


[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /mnt-homes/wapol/asoto/diegoftpxd/miniforge3/envs/openvla/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy.core.multiarray._reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch

## 5. Resultados — tabla y tasa de éxito


In [ ]:
import pandas as pd
from core import summarize

df = pd.DataFrame(results)
resumen = summarize(results)
print("Tasa de exito global: {exitos}/{total} = {tasa_exito:.0%}".format(**resumen))
df[["scenario", "instruction", "episode", "success", "steps"]]


In [ ]:
import matplotlib.pyplot as plt

por_escenario = df.groupby("scenario")["success"].mean()
ax = por_escenario.plot(kind="bar", ylim=(0, 1), color="#F58518", rot=0)
ax.set_xlabel("escenario"); ax.set_ylabel("tasa de exito")
ax.set_title("Exito por tarea (pi0)"); plt.tight_layout(); plt.show()


## 6. Resultados — videos


In [ ]:
from IPython.display import Video, display

for r in results:
    if r["video"]:
        print(f"escenario {r['scenario']} | {r['instruction']} | exito={r['success']}")
        display(Video(r["video"], embed=True, width=320))
